# PeMS 5分钟交通流数据整理与统计分析

本 notebook 实现：
1. 读取并合并多个 5 分钟交通流数据文件
2. 保留核心字段（去除元数据中已有的冗余字段）
3. 统计分析：缺失率、数据分布、异常值等

## 1. 环境配置

In [1]:
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("库加载完成！")

库加载完成！


## 2. 配置路径

In [2]:
# 数据文件夹路径
DATA_FOLDER = "/data/yuzhang_fei/PEMS/PEMSD3_2025"  # 修改为你的路径

# 输出目录
OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 查看文件列表
txt_files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.txt")))
print(f"找到 {len(txt_files)} 个数据文件")
if txt_files:
    print(f"示例: {txt_files[:3]}")

找到 363 个数据文件
示例: ['/data/yuzhang_fei/PEMS/PEMSD3_2025/d03_text_station_5min_2025_01_01.txt', '/data/yuzhang_fei/PEMS/PEMSD3_2025/d03_text_station_5min_2025_01_02.txt', '/data/yuzhang_fei/PEMS/PEMSD3_2025/d03_text_station_5min_2025_01_03.txt']


## 3. 定义数据字段

In [3]:
# 最大车道数（根据数据调整）
MAX_LANES = 8

# 基础字段
base_columns = [
    'Timestamp',      # 时间戳
    'Station',        # 站点ID
    'District',       # 区域（元数据有，可忽略）
    'Freeway',        # 高速编号（元数据有，可忽略）
    'Direction',      # 方向（元数据有，可忽略）
    'Lane_Type',      # 车道类型（元数据有，可忽略）
    'Station_Length', # 站点长度（元数据有，可忽略）
    'Samples',        # 样本数
    'Pct_Observed',   # 观测百分比
    'Total_Flow',     # 总流量
    'Avg_Occupancy',  # 平均占有率
    'Avg_Speed'       # 平均速度
]

# 车道字段
lane_columns = []
for i in range(1, MAX_LANES + 1):
    lane_columns.extend([
        f'Lane{i}_Samples',
        f'Lane{i}_Flow',
        f'Lane{i}_Avg_Occ',
        f'Lane{i}_Avg_Speed',
        f'Lane{i}_Observed'
    ])

all_columns = base_columns + lane_columns
print(f"总字段数: {len(all_columns)}")
print(f"基础字段: {len(base_columns)}")
print(f"车道字段: {len(lane_columns)} ({MAX_LANES} 车道 x 5 字段)")

总字段数: 52
基础字段: 12
车道字段: 40 (8 车道 x 5 字段)


## 4. 读取并合并数据

In [ ]:
def load_5min_data(file_path, columns):
    """
    读取单个 5 分钟数据文件
    """
    try:
        df = pd.read_csv(
            file_path,
            header=None,
            names=columns[:],  # 可能列数不完全匹配
            dtype={'Station': str},
            low_memory=False
        )
        return df
    except Exception as e:
        print(f"读取失败 {file_path}: {e}")
        return None


def load_all_data(file_list, columns, verbose=True):
    """
    读取并合并所有数据文件
    """
    dfs = []
    for i, f in enumerate(file_list):
        df = load_5min_data(f, columns)
        if df is not None:
            # 添加文件来源（可选）
            df['Source_File'] = os.path.basename(f)
            dfs.append(df)
        
        if verbose and (i + 1) % 10 == 0:
            print(f"已加载 {i + 1}/{len(file_list)} 文件")
    
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        print(f"\n合并完成: {len(combined)} 行")
        return combined
    return None


# 加载数据（如果文件很多，可以先加载部分测试）
# df = load_all_data(txt_files[:5], all_columns)  # 测试前5个文件
df = load_all_data(txt_files, all_columns)

已加载 10/363 文件
已加载 20/363 文件
已加载 30/363 文件
已加载 40/363 文件
已加载 50/363 文件
已加载 60/363 文件
已加载 70/363 文件
已加载 80/363 文件
已加载 90/363 文件
已加载 100/363 文件
已加载 110/363 文件
已加载 120/363 文件
已加载 130/363 文件
已加载 140/363 文件
已加载 150/363 文件
已加载 160/363 文件
已加载 170/363 文件


In [ ]:
# 查看数据基本信息
print("数据形状:", df.shape)
print("\n数据类型:")
print(df.dtypes)
print("\n数据预览:")
display(df.head())

## 5. 数据清洗：保留核心字段

In [ ]:
# 元数据中已有的字段（可忽略）
meta_columns = ['District', 'Freeway', 'Direction', 'Lane_Type', 'Station_Length']

# 保留的核心字段
core_columns = ['Timestamp', 'Station', 'Samples', 'Pct_Observed', 
                'Total_Flow', 'Avg_Occupancy', 'Avg_Speed']

# 车道级别字段（可选保留）
lane_core_columns = []
for i in range(1, MAX_LANES + 1):
    lane_core_columns.extend([
        f'Lane{i}_Flow',
        f'Lane{i}_Avg_Occ',
        f'Lane{i}_Avg_Speed',
        f'Lane{i}_Observed'
    ])

# 创建精简版数据（不含车道详情）
df_core = df[core_columns].copy()

# 创建完整版数据（含车道详情）
keep_columns = core_columns + [c for c in lane_core_columns if c in df.columns]
df_full = df[keep_columns].copy()

print(f"精简版字段数: {len(df_core.columns)}")
print(f"完整版字段数: {len(df_full.columns)}")

In [ ]:
# 转换时间戳
df_core['Timestamp'] = pd.to_datetime(df_core['Timestamp'], format='%m/%d/%Y %H:%M:%S')

# 转换数值类型
numeric_cols = ['Samples', 'Pct_Observed', 'Total_Flow', 'Avg_Occupancy', 'Avg_Speed']
for col in numeric_cols:
    df_core[col] = pd.to_numeric(df_core[col], errors='coerce')

print("数据类型转换完成")
print(df_core.dtypes)

## 6. 统计分析

### 6.1 缺失率分析

In [ ]:
def analyze_missing(df):
    """
    分析缺失率
    """
    missing_stats = pd.DataFrame({
        '缺失数': df.isnull().sum(),
        '缺失率(%)': (df.isnull().sum() / len(df) * 100).round(2),
        '有效数': df.notnull().sum(),
        '有效率(%)': (df.notnull().sum() / len(df) * 100).round(2)
    })
    return missing_stats.sort_values('缺失率(%)', ascending=False)


missing_stats = analyze_missing(df_core)
print("=" * 50)
print("缺失率统计")
print("=" * 50)
display(missing_stats)

In [ ]:
# 可视化缺失率
fig, ax = plt.subplots(figsize=(10, 5))
missing_pct = df_core.isnull().sum() / len(df_core) * 100
missing_pct.plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Missing Rate (%)')
ax.set_title('Missing Rate by Field')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
for i, v in enumerate(missing_pct):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'missing_rate.png'), dpi=150)
plt.show()

### 6.2 基础统计描述

In [ ]:
# 数值字段统计描述
print("=" * 50)
print("数值字段统计描述")
print("=" * 50)
display(df_core[numeric_cols].describe().round(2))

In [ ]:
# 按站点统计
station_stats = df_core.groupby('Station').agg({
    'Timestamp': 'count',
    'Total_Flow': ['mean', 'std', 'max'],
    'Avg_Speed': ['mean', 'std'],
    'Avg_Occupancy': ['mean', 'max'],
    'Pct_Observed': 'mean'
}).round(2)

station_stats.columns = ['记录数', '平均流量', '流量标准差', '最大流量', 
                          '平均速度', '速度标准差', '平均占有率', '最大占有率', '平均观测率']

print(f"\n站点数: {len(station_stats)}")
print("\n站点统计预览:")
display(station_stats.head(10))

### 6.3 时间分布分析

In [ ]:
# 提取时间特征
df_core['Date'] = df_core['Timestamp'].dt.date
df_core['Hour'] = df_core['Timestamp'].dt.hour
df_core['DayOfWeek'] = df_core['Timestamp'].dt.dayofweek  # 0=Monday

# 按日期统计记录数
daily_counts = df_core.groupby('Date').size()
print(f"日期范围: {daily_counts.index.min()} ~ {daily_counts.index.max()}")
print(f"总天数: {len(daily_counts)}")
print(f"每日平均记录数: {daily_counts.mean():.0f}")

In [ ]:
# 按小时统计平均流量
hourly_flow = df_core.groupby('Hour')['Total_Flow'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
hourly_flow.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Flow (Veh/5min)')
ax.set_title('Average Traffic Flow by Hour')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hourly_flow.png'), dpi=150)
plt.show()

In [ ]:
# 按星期统计
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly_flow = df_core.groupby('DayOfWeek')['Total_Flow'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
weekly_flow.plot(kind='bar', ax=ax, color='steelblue')
ax.set_xticklabels(day_names, rotation=0)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Flow (Veh/5min)')
ax.set_title('Average Traffic Flow by Day of Week')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'weekly_flow.png'), dpi=150)
plt.show()

### 6.4 数据质量分析

In [ ]:
# 观测率分布
print("=" * 50)
print("观测率 (Pct_Observed) 分布")
print("=" * 50)

obs_bins = [0, 25, 50, 75, 100]
obs_labels = ['0-25%', '25-50%', '50-75%', '75-100%']
df_core['Obs_Bin'] = pd.cut(df_core['Pct_Observed'], bins=obs_bins, labels=obs_labels, include_lowest=True)

obs_dist = df_core['Obs_Bin'].value_counts().sort_index()
obs_pct = (obs_dist / len(df_core) * 100).round(2)

print(pd.DataFrame({'数量': obs_dist, '占比(%)': obs_pct}))

In [ ]:
# 零值分析
print("=" * 50)
print("零值分析")
print("=" * 50)

zero_stats = pd.DataFrame({
    '零值数': (df_core[numeric_cols] == 0).sum(),
    '零值率(%)': ((df_core[numeric_cols] == 0).sum() / len(df_core) * 100).round(2)
})
display(zero_stats)

In [ ]:
# 异常值检测（使用 IQR 方法）
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = (series < lower) | (series > upper)
    return outliers.sum(), lower, upper

print("=" * 50)
print("异常值检测 (IQR 方法)")
print("=" * 50)

for col in ['Total_Flow', 'Avg_Speed', 'Avg_Occupancy']:
    n_outliers, lower, upper = detect_outliers_iqr(df_core[col].dropna())
    pct = n_outliers / df_core[col].notna().sum() * 100
    print(f"{col}: {n_outliers} 个异常值 ({pct:.2f}%), 范围: [{lower:.2f}, {upper:.2f}]")

### 6.5 数据分布可视化

In [ ]:
# 核心指标分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 流量分布
df_core['Total_Flow'].dropna().hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Total Flow (Veh/5min)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Flow Distribution')

# 速度分布
df_core['Avg_Speed'].dropna().hist(bins=50, ax=axes[1], color='green', edgecolor='white')
axes[1].set_xlabel('Avg Speed (mph)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Speed Distribution')

# 占有率分布
df_core['Avg_Occupancy'].dropna().hist(bins=50, ax=axes[2], color='orange', edgecolor='white')
axes[2].set_xlabel('Avg Occupancy')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Occupancy Distribution')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'distributions.png'), dpi=150)
plt.show()

In [ ]:
# 流量-速度关系（基本图）
sample = df_core.dropna(subset=['Total_Flow', 'Avg_Speed']).sample(min(50000, len(df_core)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sample['Total_Flow'], sample['Avg_Speed'], alpha=0.1, s=1)
ax.set_xlabel('Total Flow (Veh/5min)')
ax.set_ylabel('Avg Speed (mph)')
ax.set_title('Flow-Speed Fundamental Diagram')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'flow_speed_diagram.png'), dpi=150)
plt.show()

## 7. 生成统计报告

In [ ]:
# 汇总统计报告
report = f"""
{'='*60}
PeMS 5分钟交通流数据统计报告
{'='*60}

【数据概况】
- 数据文件数: {len(txt_files)}
- 总记录数: {len(df_core):,}
- 站点数: {df_core['Station'].nunique()}
- 日期范围: {df_core['Date'].min()} ~ {df_core['Date'].max()}
- 总天数: {df_core['Date'].nunique()}

【缺失率统计】
{missing_stats.to_string()}

【核心指标统计】
{df_core[numeric_cols].describe().round(2).to_string()}

【观测率分布】
- 0-25%: {obs_pct.get('0-25%', 0):.2f}%
- 25-50%: {obs_pct.get('25-50%', 0):.2f}%
- 50-75%: {obs_pct.get('50-75%', 0):.2f}%
- 75-100%: {obs_pct.get('75-100%', 0):.2f}%

【零值统计】
{zero_stats.to_string()}

{'='*60}
报告生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
{'='*60}
"""

print(report)

# 保存报告
with open(os.path.join(OUTPUT_DIR, 'statistics_report.txt'), 'w') as f:
    f.write(report)
print(f"\n报告已保存: {OUTPUT_DIR}/statistics_report.txt")

## 8. 保存整理后的数据

In [ ]:
# 保存精简版数据
output_columns = ['Timestamp', 'Station', 'Samples', 'Pct_Observed', 
                  'Total_Flow', 'Avg_Occupancy', 'Avg_Speed']
df_output = df_core[output_columns].copy()

# 保存为 CSV
csv_path = os.path.join(DATA_FOLDER, 'pems_5min_cleaned.csv')
df_output.to_csv(csv_path, index=False)
print(f"数据已保存: {csv_path}")
print(f"文件大小: {os.path.getsize(csv_path) / 1024 / 1024:.2f} MB")

In [ ]:
# 保存为 Parquet（更高效）
try:
    parquet_path = os.path.join(DATA_FOLDER, 'pems_5min_cleaned.parquet')
    df_output.to_parquet(parquet_path, index=False)
    print(f"Parquet 已保存: {parquet_path}")
    print(f"文件大小: {os.path.getsize(parquet_path) / 1024 / 1024:.2f} MB")
except:
    print("Parquet 保存失败，需要安装 pyarrow: pip install pyarrow")

## 9. 使用说明

### 输出文件

| 文件 | 说明 |
|------|------|
| `pems_5min_cleaned.csv` | 整理后的核心数据 |
| `pems_5min_cleaned.parquet` | Parquet 格式（更高效） |
| `statistics_report.txt` | 统计报告 |
| `missing_rate.png` | 缺失率图 |
| `hourly_flow.png` | 小时流量图 |
| `weekly_flow.png` | 星期流量图 |
| `distributions.png` | 数据分布图 |
| `flow_speed_diagram.png` | 流量-速度基本图 |

### 保留字段说明

| 字段 | 说明 | 单位 |
|------|------|------|
| Timestamp | 时间戳 | - |
| Station | 站点ID | - |
| Samples | 样本数 | - |
| Pct_Observed | 观测百分比 | % |
| Total_Flow | 总流量 | Veh/5min |
| Avg_Occupancy | 平均占有率 | 0-1 |
| Avg_Speed | 平均速度 | mph |